# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EgeGln365/FlyRank_AI_Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nCSV files found:")
for path in Path(".").rglob("*.csv"):
    print(path)

Current working directory:
/Users/egegulunay/PROJECTS/FlyRank_AI_Internship/work/notebooks

CSV files found:
work/outputs/baseline_action_score.csv


In [3]:
import pandas as pd
from pathlib import Path

baseline_path = Path("work/outputs/baseline_action_score.csv")

baseline = pd.read_csv(baseline_path)

print("Queue rows:", len(baseline))

print("\nColumns:")
print(baseline.columns.tolist())

baseline.head()

Queue rows: 14997

Columns:
['rank', 'client_hash_id', 'content_hash_id', 'action', 'reason_code', 'action_score', 'avg_position_march', 'impressions_march', 'ctr_march', 'ctr_threshold', 'ctr_gap']


,rank,client_hash_id,content_hash_id,action,reason_code,action_score,avg_position_march,impressions_march,ctr_march,ctr_threshold,ctr_gap
0,1,client_23a62021009f63c4,content_44f34c0a90047651,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,305.819876,0.665877,212404.0,0.000113,0.001553,0.001440
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,208.602484,2.693038,134984.0,0.000007,0.001553,0.001545
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,191.663043,0.308426,124075.0,0.000008,0.001553,0.001545
3,4,client_62f4a7e64f5e0096,content_34a70fea29d15f24,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,183.295886,3.166132,143019.0,0.000301,0.001582,0.001282
4,5,client_62f4a7e64f5e0096,content_f6116743b00afc2d,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,155.227848,9.735658,107584.0,0.000139,0.001582,0.001443


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Action logic

The action queue uses the validated Week-4 rule-based baseline as its primary ranking method. In the Week-5 client-grouped evaluation, this baseline produced stronger Precision@20 and Precision@50 than the Logistic Regression model, so the playbook keeps the simpler and more interpretable rule as the main decision-support mechanism.

The primary archetype is **HIGH_VISIBILITY_LOW_CTR**: content that already has meaningful search visibility and a good average search position, but whose CTR is low relative to its position bucket.

For these pages:

- **Action:** `CTR_FIX`
- **Reason code:** `LOW_CTR_HIGH_VISIBILITY`
- **Suggested review:** inspect the title, meta description, SERP presentation, and search-intent alignment.
- **Priority:** use `action_score = ctr_gap × impressions_march` to rank larger observed CTR opportunities first.

The queue is a prioritization tool for human review. A high rank does not mean that a page should be changed automatically or that a CTR improvement is guaranteed.

In [4]:
print(baseline.columns.tolist())

['rank', 'client_hash_id', 'content_hash_id', 'action', 'reason_code', 'action_score', 'avg_position_march', 'impressions_march', 'ctr_march', 'ctr_threshold', 'ctr_gap']


In [5]:
action_queue = baseline.copy()

action_queue["archetype"] = "HIGH_VISIBILITY_LOW_CTR"

action_queue["recommended_review"] = (
    "Review title, meta description, SERP presentation, "
    "and search-intent alignment"
)

action_queue = action_queue.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

action_queue["rank"] = range(1, len(action_queue) + 1)

action_queue[[
    "rank",
    "client_hash_id",
    "content_hash_id",
    "archetype",
    "action",
    "reason_code",
    "action_score",
    "avg_position_march",
    "impressions_march",
    "ctr_march",
    "ctr_threshold",
    "ctr_gap",
    "recommended_review"
]].head(10)

,rank,client_hash_id,content_hash_id,archetype,action,reason_code,action_score,avg_position_march,impressions_march,ctr_march,ctr_threshold,ctr_gap,recommended_review
0,1,client_23a62021009f63c4,content_44f34c0a90047651,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,305.819876,0.665877,212404.0,0.000113,0.001553,0.001440,"Review title, meta description, SERP presentat..."
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,208.602484,2.693038,134984.0,0.000007,0.001553,0.001545,"Review title, meta description, SERP presentat..."
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,191.663043,0.308426,124075.0,0.000008,0.001553,0.001545,"Review title, meta description, SERP presentat..."
3,4,client_62f4a7e64f5e0096,content_34a70fea29d15f24,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,183.295886,3.166132,143019.0,0.000301,0.001582,0.001282,"Review title, meta description, SERP presentat..."
4,5,client_62f4a7e64f5e0096,content_f6116743b00afc2d,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,155.227848,9.735658,107584.0,0.000139,0.001582,0.001443,"Review title, meta description, SERP presentat..."
5,6,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,137.348101,7.831807,89332.0,0.000045,0.001582,0.001538,"Review title, meta description, SERP presentat..."
6,7,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,129.177019,0.116003,83834.0,0.000012,0.001553,0.001541,"Review title, meta description, SERP presentat..."
7,8,client_62f4a7e64f5e0096,content_7c6373141eae744a,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,126.799051,5.948459,132593.0,0.000626,0.001582,0.000956,"Review title, meta description, SERP presentat..."
8,9,client_a80fca3f171ed1de,content_046fc480045b88f5,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,126.575949,7.208276,83788.0,0.000072,0.001582,0.001511,"Review title, meta description, SERP presentat..."
9,10,client_a80fca3f171ed1de,content_9540d884af3e41fd,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,119.341772,8.005184,82376.0,0.000134,0.001582,0.001449,"Review title, meta description, SERP presentat..."


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## Intended use

This playbook is designed as a decision-support tool for SEO and content teams. Its purpose is to prioritize which content pages should be reviewed first rather than requiring teams to manually inspect the full content portfolio.

The ranked queue highlights pages with meaningful search visibility, relatively good search positions, and low CTR relative to their position range. A `CTR_FIX` recommendation means that the page should be reviewed for possible improvements to its title, meta description, SERP presentation, and search-intent alignment.

The ranking is intended to support human prioritization. It does not automatically determine whether a page should be changed.

## Limits

The recommendations are based primarily on observed GSC performance signals and should not be interpreted as causal evidence that a specific content change will improve CTR or traffic.

The validation was performed on a retrospective March-to-April 2026 evaluation population with sufficient GSC coverage. Therefore, the measured performance should not be assumed to generalize equally to every client, content type, or future time period.

The `action_score` represents prioritization based on the observed CTR gap and impression volume. It is not an estimate of guaranteed additional clicks, revenue, or business value.

Content context such as seasonality, business importance, search intent changes, brand requirements, and editorial quality is not fully represented by the ranking and should be considered during human review.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## Human review rules

Every recommendation in the ranked queue requires human review before any content change is made.

For a `CTR_FIX` recommendation, the reviewer should check:

- whether the page still matches the intended search intent;
- whether the title and meta description accurately represent the content;
- whether low CTR may be explained by seasonality, query mix, brand constraints, or other context not captured by the ranking;
- whether the page is strategically important or has editorial requirements that should override the recommendation;
- whether a content refresh is more appropriate than a simple SERP presentation change.

Observed performance decay can also be used as a signal for refresh review. However, declining performance alone is not evidence that refreshing the page will improve future performance. It should trigger investigation rather than an automatic refresh.

## No-go list

The playbook should **not** automatically:

- rewrite or publish titles, meta descriptions, or page content;
- delete, redirect, unpublish, or substantially modify pages;
- trigger a content refresh only because performance declined;
- treat `action_score` as guaranteed traffic, click, or revenue gain;
- act on pages without checking business, editorial, and search-intent context;
- replace final editorial or SEO decisions made by a human reviewer.

The system ranks potential opportunities; it does not execute content changes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## Monitoring and retrain triggers

This playbook is intended as a lightweight decision-support workflow rather than a production monitoring system.

The ranking logic should be reviewed when:

- Precision@20 or Precision@50 drops meaningfully on a new evaluation period;
- the distribution of important signals such as CTR, impressions, or average position changes substantially;
- the proportion of content receiving a recommendation changes substantially;
- GSC data availability or data quality changes;
- the system is applied to new clients or content types that were not represented in the validation data.

A retraining or rule-review step should be considered when these changes persist across evaluation periods rather than reacting to a single short-term fluctuation.

Performance should continue to be evaluated with client-grouped validation when the goal is to estimate generalization to unseen clients.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
from pathlib import Path
import pandas as pd

# ---------------------------------------------------------
# 1. Define the correct repository output directory
# ---------------------------------------------------------

# This notebook lives in:
# work/notebooks/w07_action_playbook.ipynb
#
# Therefore ../outputs points to:
# work/outputs/

OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

QUEUE_PATH = OUTPUT_DIR / "w07_ranked_action_queue.csv"


# ---------------------------------------------------------
# 2. Define the columns that belong in the paper export
# ---------------------------------------------------------

export_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "archetype",
    "action",
    "reason_code",
    "action_score",
    "avg_position_march",
    "impressions_march",
    "ctr_march",
    "ctr_threshold",
    "ctr_gap",
    "recommended_review",
]


# ---------------------------------------------------------
# 3. Validate the queue before exporting
# ---------------------------------------------------------

missing_columns = [
    col for col in export_columns
    if col not in action_queue.columns
]

if missing_columns:
    raise ValueError(
        f"Cannot export queue. Missing columns: {missing_columns}"
    )

if action_queue.empty:
    raise ValueError("Cannot export queue: action_queue is empty.")

if action_queue["rank"].duplicated().any():
    raise ValueError("Cannot export queue: duplicate ranks found.")

if action_queue["action_score"].isna().any():
    raise ValueError(
        "Cannot export queue: missing action_score values found."
    )

# Make sure the queue is actually ordered by priority.
final_queue = (
    action_queue[export_columns]
    .sort_values("action_score", ascending=False)
    .reset_index(drop=True)
    .copy()
)

# Recreate rank after sorting so rank always matches export order.
final_queue["rank"] = range(1, len(final_queue) + 1)


# ---------------------------------------------------------
# 4. Export the ranked action queue
# ---------------------------------------------------------

final_queue.to_csv(
    QUEUE_PATH,
    index=False
)


# ---------------------------------------------------------
# 5. Verify that the exported file can be read back
# ---------------------------------------------------------

export_check = pd.read_csv(QUEUE_PATH)

assert len(export_check) == len(final_queue), (
    "Exported row count does not match final_queue."
)

assert export_check["rank"].iloc[0] == 1, (
    "Exported queue does not start at rank 1."
)

assert export_check["rank"].iloc[-1] == len(export_check), (
    "Exported ranks are incomplete."
)

assert export_check["action_score"].is_monotonic_decreasing, (
    "Exported queue is not sorted by action_score."
)


# ---------------------------------------------------------
# 6. Print a compact export receipt
# ---------------------------------------------------------

print("W07 ACTION PLAYBOOK EXPORT")
print("-" * 40)
print(f"Rows exported : {len(export_check):,}")
print(f"Unique clients: {export_check['client_hash_id'].nunique():,}")
print(f"Actions       : {export_check['action'].value_counts().to_dict()}")
print(f"Reason codes  : {export_check['reason_code'].value_counts().to_dict()}")
print(f"Top score     : {export_check['action_score'].max():.6f}")
print(f"Lowest score  : {export_check['action_score'].min():.6f}")
print(f"Output path   : {QUEUE_PATH.resolve()}")

export_check.head(10)

W07 ACTION PLAYBOOK EXPORT
----------------------------------------
Rows exported : 14,997
Unique clients: 24
Actions       : {'CTR_FIX': 14997}
Reason codes  : {'LOW_CTR_HIGH_VISIBILITY': 14997}
Top score     : 305.819876
Lowest score  : 0.000000
Output path   : /Users/egegulunay/PROJECTS/FlyRank_AI_Internship/work/outputs/w07_ranked_action_queue.csv


,rank,client_hash_id,content_hash_id,archetype,action,reason_code,action_score,avg_position_march,impressions_march,ctr_march,ctr_threshold,ctr_gap,recommended_review
0,1,client_23a62021009f63c4,content_44f34c0a90047651,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,305.819876,0.665877,212404.0,0.000113,0.001553,0.001440,"Review title, meta description, SERP presentat..."
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,208.602484,2.693038,134984.0,0.000007,0.001553,0.001545,"Review title, meta description, SERP presentat..."
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,191.663043,0.308426,124075.0,0.000008,0.001553,0.001545,"Review title, meta description, SERP presentat..."
3,4,client_62f4a7e64f5e0096,content_34a70fea29d15f24,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,183.295886,3.166132,143019.0,0.000301,0.001582,0.001282,"Review title, meta description, SERP presentat..."
4,5,client_62f4a7e64f5e0096,content_f6116743b00afc2d,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,155.227848,9.735658,107584.0,0.000139,0.001582,0.001443,"Review title, meta description, SERP presentat..."
5,6,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,137.348101,7.831807,89332.0,0.000045,0.001582,0.001538,"Review title, meta description, SERP presentat..."
6,7,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,129.177019,0.116003,83834.0,0.000012,0.001553,0.001541,"Review title, meta description, SERP presentat..."
7,8,client_62f4a7e64f5e0096,content_7c6373141eae744a,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,126.799051,5.948459,132593.0,0.000626,0.001582,0.000956,"Review title, meta description, SERP presentat..."
8,9,client_a80fca3f171ed1de,content_046fc480045b88f5,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,126.575949,7.208276,83788.0,0.000072,0.001582,0.001511,"Review title, meta description, SERP presentat..."
9,10,client_a80fca3f171ed1de,content_9540d884af3e41fd,HIGH_VISIBILITY_LOW_CTR,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,119.341772,8.005184,82376.0,0.000134,0.001582,0.001449,"Review title, meta description, SERP presentat..."


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.